In [46]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

# Conditional Chain — Routing to Different Chains Based on Content

## What is a Conditional Chain?

A **conditional chain** (also called a **branch chain**) routes the input to different chains depending on some condition — like an `if/else` in regular code, but for LLM pipelines.

```
                    ┌→ thank_you_chain (if positive)
input → classify ──┤
                    └→ improvement_chain (if negative)
```

This is incredibly useful for building smart pipelines that behave differently based on the nature of the input.

## Real-world examples

- **Customer feedback router**: positive feedback → thank-you email, negative → support ticket
- **Question classifier**: simple question → FAQ lookup, complex question → LLM response
- **Language detector**: English → English chain, French → French chain
- **Intent router**: "Book a flight" → booking chain, "Cancel flight" → cancellation chain

## The key components

### 1. `RunnableBranch`
Routes the input to the first branch whose condition evaluates to `True`.

```python
branch = RunnableBranch(
    (condition_1, chain_1),   # if condition_1 is True → run chain_1
    (condition_2, chain_2),   # elif condition_2 is True → run chain_2
    default_chain             # else → always required, runs if no condition matched
)
```

> **Important:** `RunnableBranch` **always requires a default** as the last argument. Omitting it raises a `TypeError`.

### 2. `PydanticOutputParser`
Parses the model's JSON output into a Pydantic object. Used here to get a typed `Feedback` object with `.feedback == "positive"` or `"negative"`.

## What this notebook builds

1. A **classifier chain** that reads customer feedback and returns `positive` or `negative`
2. A **positive response chain** that writes a thank-you message
3. A **negative response chain** that generates improvement suggestions
4. A **full chain** that classifies first, then routes to the appropriate response

## What you'll learn

- How `RunnableBranch` creates conditional routing
- How `PydanticOutputParser` turns raw text into a typed object
- How to build a multi-step classify-then-respond pipeline
- Why a default branch is always required

## Prerequisites

- Ollama running with a model pulled (uses `deepseek-v3.1:671b-cloud` — change to any local model)
- Virtual environment activated

In [47]:
model = ChatOllama(model="deepseek-v3.1:671b-cloud")

In [48]:
class Feedback(BaseModel):
    feedback: Literal["positive", "negative"] = Field(
        description="The feedback can be either positive or negative."
    )

In [49]:
parser = PydanticOutputParser(pydantic_object=Feedback)


In [50]:
prompt = PromptTemplate(
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
    template=(
        "You are a sentiment analysis assistant.\n"
        "Classify the sentiment of the following text as positive or negative.\n\n"
        "Text:\n{feedback}\n\n"
        "Follow these instructions when formatting your output:\n"
        "{format_instructions}"
    ),
)

In [51]:
prompt

PromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"feedback": {"description": "The feedback can be either positive or negative.", "enum": ["positive", "negative"], "title": "Feedback", "type": "string"}}, "required": ["feedback"]}\n```'}, template='You are a sentiment analysis assistant.\nClassify the sentiment of the following text as positive or negative.\n\nText:\n{feedback}\n\nFollow these instructions when formatting your output:\n{format_instructions}')

In [52]:
chain  = prompt | model | parser

In [53]:
chain.invoke({"feedback": "This product is amazing! I love it."})

Feedback(feedback='positive')

In [ ]:
# prompt for when the feedback is POSITIVE — write a thank-you
positive_prompt = PromptTemplate(
    input_variables=["feedback"],
    template=(
        "You are a customer success manager.\n"
        "A customer left this positive review:\n\n{feedback}\n\n"
        "Write a warm, personalised thank-you reply in 2-3 sentences."
    ),
)

# prompt for when the feedback is NEGATIVE — give improvement suggestions
negative_prompt = PromptTemplate(
    input_variables=["feedback"],
    template=(
        "You are a product improvement specialist.\n"
        "A customer left this negative review:\n\n{feedback}\n\n"
        "List 3 concrete, actionable suggestions to address the concerns raised."
    ),
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Step 1: classify sentiment — takes {"feedback": text}, returns Feedback(feedback='positive'/'negative')
classify_chain = prompt | model | parser

# Step 2: build a response chain for each sentiment class
positive_response_chain = positive_prompt | model | StrOutputParser()
negative_response_chain = negative_prompt | model | StrOutputParser()

# Step 3: run classifier and keep the original feedback text alongside the result
def classify_and_pass(x):
    sentiment = classify_chain.invoke(x)
    return {"feedback": x["feedback"], "sentiment": sentiment}

# Step 4: route to the correct chain based on the classified sentiment
# RunnableBranch always needs a default as the final argument — it's required, not optional
branch_chain = RunnableBranch(
    (
        lambda x: x["sentiment"].feedback == "positive",
        RunnableLambda(lambda x: {"feedback": x["feedback"]}) | positive_response_chain,
    ),
    (
        lambda x: x["sentiment"].feedback == "negative",
        RunnableLambda(lambda x: {"feedback": x["feedback"]}) | negative_response_chain,
    ),
    RunnableLambda(lambda x: f"Received feedback: {x['feedback']}"),  # required default
)

full_chain = RunnableLambda(classify_and_pass) | branch_chain

# Test with positive feedback
print("=== Positive Feedback ===")
print(full_chain.invoke({"feedback": "This product is amazing! I absolutely love it!"}))

print("\n=== Negative Feedback ===")
print(full_chain.invoke({"feedback": "Terrible quality — it broke after just one day of use."}))

In [ ]:
# --- Summary ---
# The full_chain built in this notebook:
#
# User text (e.g. "Amazing product!")
#     ↓
# classify_chain  → Feedback(feedback='positive')
#     ↓
# classify_and_pass  → {"feedback": "Amazing product!", "sentiment": Feedback(...)}
#     ↓
# RunnableBranch
#     ├── if positive → positive_prompt | model → "Thank you for your kind words!"
#     └── if negative → negative_prompt | model → "Here are 3 improvement suggestions..."

print("Conditional chain demo complete.")
print("Key concepts covered:")
print("  - RunnableBranch for conditional routing")
print("  - PydanticOutputParser for typed sentiment classification")
print("  - RunnableLambda for custom Python functions inside a chain")
print("  - Full classify-then-respond pipeline")